# Evaluation Notebook - LoRA Training + Runtime Generation

This notebook evaluates your improved **Serializing Java Objects in Plain Code** pipeline with **LLM training and runtime generation**.

It includes:
- Dataset quality and token-length graphics
- Training dynamics graphics (loss + learning rate)
- Baseline vs LoRA output comparison
- Optional runtime metrics visualization from CSV logs

In [ ]:
# If needed, install deps in Colab/Jupyter
%pip install -q transformers peft datasets sentencepiece matplotlib seaborn pandas

In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "./my-testgen-lora"   # local adapter folder after training
HF_ADAPTER_REPO = None                   # or set e.g. "your-hf-username/my-testgen-lora"
DATASET_NAME = "zzzghttt/context2test"
DATASET_SPLIT = "train"
EVAL_SAMPLES = 40
SEED = 42

np.random.seed(SEED)

In [ ]:
PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_prompt(context: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip())

def has_assertion(test: str) -> bool:
    return any(k in test for k in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    opens = code.count("{")
    closes = code.count("}")
    return opens > 0 and abs(opens - closes) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

raw = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
col_context = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
col_test = next((c for c in ["test", "output", "target"] if c in raw.column_names), None)

if col_context is None or col_test is None:
    raise ValueError(f"Could not find context/test columns in {raw.column_names}")

df = raw.to_pandas()[[col_context, col_test]].rename(columns={col_context: "context", col_test: "reference_test"})

df = df.drop_duplicates(subset=["context"]).copy()
df = df[df["reference_test"].apply(has_assertion)]
df = df[df["reference_test"].apply(is_valid_java)]
df = df[df["context"].str.strip().str.len() > 30]

df["prompt"] = df["context"].apply(build_prompt)
df["prompt_token_est"] = df["prompt"].apply(estimate_tokens)
df["ref_token_est"] = df["reference_test"].apply(estimate_tokens)
df["assert_count"] = df["reference_test"].str.count(r"\bassert\w*")

eval_df = df.sample(n=min(EVAL_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)

print(f"Total cleaned rows: {len(df)}")
print(f"Evaluation rows sampled: {len(eval_df)}")
display(eval_df[["context", "reference_test", "prompt_token_est", "ref_token_est"]].head(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["prompt_token_est"], bins=40, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Prompt Length Distribution (Estimated Tokens)")
axes[0].set_xlabel("Estimated prompt tokens")

sns.histplot(df["ref_token_est"], bins=40, kde=True, ax=axes[1], color="#2ca02c")
axes[1].set_title("Reference Test Length Distribution (Estimated Tokens)")
axes[1].set_xlabel("Estimated reference-test tokens")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=df[["prompt_token_est", "ref_token_est"]], palette=["#1f77b4", "#2ca02c"])
plt.title("Token Length Boxplot")
plt.ylabel("Estimated tokens")
plt.show()

In [ ]:
# Load trainer_state.json from output folder or checkpoints and plot learning dynamics
def find_trainer_state_files(root_dir: str):
    root = Path(root_dir)
    if not root.exists():
        return []
    files = list(root.rglob("trainer_state.json"))
    return sorted(files, key=lambda p: len(str(p)))

state_files = find_trainer_state_files(ADAPTER_PATH)
print("Found trainer_state files:")
for f in state_files:
    print(" -", f)

if state_files:
    chosen = state_files[-1]
    with open(chosen, "r", encoding="utf-8") as fh:
        trainer_state = json.load(fh)

    logs = pd.DataFrame(trainer_state.get("log_history", []))
    train_logs = logs[logs["loss"].notna()].copy() if "loss" in logs.columns else pd.DataFrame()

    if not train_logs.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        sns.lineplot(data=train_logs, x="step", y="loss", marker="o", ax=axes[0], color="#d62728")
        axes[0].set_title("Training Loss vs Step")
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")

        if "learning_rate" in train_logs.columns:
            sns.lineplot(data=train_logs, x="step", y="learning_rate", marker="o", ax=axes[1], color="#9467bd")
            axes[1].set_title("Learning Rate Schedule")
            axes[1].set_xlabel("Step")
            axes[1].set_ylabel("Learning rate")
        else:
            axes[1].axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No training loss logs found in trainer_state.json")
else:
    print("No trainer_state.json found. Run training first or set ADAPTER_PATH correctly.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

adapter_source = HF_ADAPTER_REPO or ADAPTER_PATH
if not Path(ADAPTER_PATH).exists() and HF_ADAPTER_REPO is None:
    raise FileNotFoundError("Adapter not found locally. Set ADAPTER_PATH or HF_ADAPTER_REPO.")

lora_model = PeftModel.from_pretrained(baseline_model, adapter_source)
lora_model.eval()
baseline_model.eval()

print("Models loaded successfully")

In [ ]:
def generate(model, prompt: str, max_new_tokens: int = 200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded

def lexical_f1(reference: str, prediction: str) -> float:
    ref_tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", reference)
    pred_tokens = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", prediction)
    if not ref_tokens or not pred_tokens:
        return 0.0

    ref_set = set(ref_tokens)
    pred_set = set(pred_tokens)
    tp = len(ref_set & pred_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(ref_set) if ref_set else 0.0
    return 2 * precision * recall / (precision + recall + 1e-8)

def quality_features(java_text: str) -> dict:
    return {
        "has_test_annotation": int("@Test" in java_text),
        "assert_count": len(re.findall(r"\bassert\w*", java_text)),
        "brace_balance": int(java_text.count("{") == java_text.count("}")),
        "line_count": len(java_text.splitlines())
    }

rows = []
for i, r in eval_df.iterrows():
    prompt = r["prompt"]
    ref = r["reference_test"]

    pred_base = generate(baseline_model, prompt)
    pred_lora = generate(lora_model, prompt)

    base_feats = quality_features(pred_base)
    lora_feats = quality_features(pred_lora)

    rows.append({
        "sample_id": i,
        "f1_base": lexical_f1(ref, pred_base),
        "f1_lora": lexical_f1(ref, pred_lora),
        "asserts_base": base_feats["assert_count"],
        "asserts_lora": lora_feats["assert_count"],
        "balanced_base": base_feats["brace_balance"],
        "balanced_lora": lora_feats["brace_balance"],
        "has_test_base": base_feats["has_test_annotation"],
        "has_test_lora": lora_feats["has_test_annotation"],
        "pred_base": pred_base,
        "pred_lora": pred_lora,
        "reference": ref
    })

results = pd.DataFrame(rows)
display(results.head(3))

In [ ]:
summary = pd.DataFrame({
    "metric": ["Lexical F1", "Assert Count", "Brace Balance Rate", "@Test Presence Rate"],
    "baseline": [
        results["f1_base"].mean(),
        results["asserts_base"].mean(),
        results["balanced_base"].mean(),
        results["has_test_base"].mean()
    ],
    "lora": [
        results["f1_lora"].mean(),
        results["asserts_lora"].mean(),
        results["balanced_lora"].mean(),
        results["has_test_lora"].mean()
    ]
})

display(summary)

plot_df = summary.melt(id_vars="metric", var_name="model", value_name="value")
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette=["#1f77b4", "#ff7f0e"])
plt.title("Baseline vs LoRA Quality Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

delta = (summary["lora"] - summary["baseline"]).rename("lora_minus_baseline")
display(pd.concat([summary[["metric"]], delta], axis=1))

## Optional: Runtime Metrics (from CSV)

If you collect runtime generation logs (for example compile success, test pass, latency), place a CSV and set `RUNTIME_CSV_PATH` below.

Expected columns (adaptable):
- `model`: baseline or lora
- `compile_success`: 0/1
- `test_pass`: 0/1
- `latency_ms`: numeric

In [ ]:
RUNTIME_CSV_PATH = None  # e.g. './runtime_eval_results.csv'

if RUNTIME_CSV_PATH and Path(RUNTIME_CSV_PATH).exists():
    rt = pd.read_csv(RUNTIME_CSV_PATH)
    display(rt.head())

    metric_cols = [c for c in ["compile_success", "test_pass", "latency_ms"] if c in rt.columns]
    if "model" not in rt.columns:
        raise ValueError("Runtime CSV must include a 'model' column.")

    agg = rt.groupby("model", as_index=False)[metric_cols].mean(numeric_only=True)
    display(agg)

    for col in metric_cols:
        plt.figure(figsize=(6, 4))
        sns.barplot(data=agg, x="model", y=col, palette="Set2")
        plt.title(f"Runtime Metric: {col}")
        plt.tight_layout()
        plt.show()
else:
    print("No runtime CSV provided yet. Skipping runtime plots.")